# Block 1: Data Cleaning and Label Creation

This notebook is the official starting point for the EventOps AI pipeline.

**Goal:** Convert the raw Astram traffic event CSV into a cleaned, label-ready dataset without leaking future information into model features.

Outputs:

- `outputs/cleaned/cleaned_events_with_labels.csv`
- `outputs/cleaned/label_summary.csv`
- `outputs/cleaned/leakage_policy.csv`

The labels created here are used by later notebooks:

- Model 1: `target_road_closure`
- Model 2: `duration_band`

## 1. Imports and Project Paths

This resolves the repository root automatically, so the notebook works from VS Code, Jupyter, or Colab after the project folder is available.

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 100)

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'data').exists()), cwd)
DATA_PATH = PROJECT_ROOT / 'data' / 'Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'cleaned'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Raw data path:', DATA_PATH)
print('Output dir:', OUTPUT_DIR)
print('Raw data exists:', DATA_PATH.exists())

Project root: /Users/astron_designer/GridLock_Phase2
Raw data path: /Users/astron_designer/GridLock_Phase2/data/Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv
Output dir: /Users/astron_designer/GridLock_Phase2/outputs/cleaned
Raw data exists: True


## 2. Load Raw Dataset

The raw dataset has one row per reported traffic event.

In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
print('Raw shape:', raw_df.shape)
display(raw_df.head())

Raw shape: (8173, 46)


,id,event_type,latitude,longitude,endlatitude,endlongitude,address,end_address,event_cause,requires_road_closure,start_datetime,end_datetime,status,authenticated,modified_datetime,map_file,direction,description,veh_type,veh_no,corridor,priority,cargo_material,reason_breakdown,age_of_truck,created_date,route_path,client_id,created_by_id,last_modified_by_id,assigned_to_police_id,citizen_accident_id,comment,police_station,meta_data,kgid,resolved_at_address,resolved_at_latitude,resolved_at_longitude,closed_by_id,closed_datetime,resolved_by_id,resolved_datetime,gba_identifier,zone,junction
0,FKID000000,unplanned,13.040004,77.518099,0.000000,0.000000,"Mumbai Bengaluru Highway, Jalahalli Cross Junc...",NaN,vehicle_breakdown,False,2024-03-07 17:01:48.111+00,NaN,closed,yes,2024-03-07 19:35:47.871698+00,NaN,NaN,s m circle in coming man track,lcv,FKN00GL0000,Tumkur Road,High,NaN,NaN,NaN,2024-03-07 17:03:51.164032+00,NaN,1,FKUSR00000,FKUSR00001,NaN,NaN,NaN,Peenya,NaN,FKKG000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,FKID000001,unplanned,12.921876,77.645158,0.000000,0.000000,"19th Main Road, Heavie Halcyon, Agara, HSR Lay...",NaN,vehicle_breakdown,False,2024-01-30 04:07:24.173+00,NaN,resolved,yes,2024-01-30 04:17:46.828979+00,NaN,NaN,Starting problem,heavy_vehicle,FKN00GL0001,ORR East 1,High,NaN,NaN,NaN,2024-01-30 04:08:22.954979+00,NaN,1,FKUSR00002,FKUSR00001,NaN,NaN,NaN,HSR Layout,NaN,FKKG000001,"19th Main Road, Heavie Halcyon, Agara, HSR Lay...",12.921876,77.645158,NaN,NaN,FKUSR00002,2024-01-30 04:17:46.828355+00,NaN,NaN,NaN
2,FKID000002,unplanned,12.955622,77.585708,0.000000,0.000000,"Lalbagh Main Road, Dr Sri Shantaveera Swami Ci...",NaN,others,False,2023-11-11 06:18:03.343+00,NaN,closed,yes,2024-01-30 04:56:03.282003+00,NaN,NaN,ಊರ್ವಶಿ ಜಂಕ್ಷನ್ ನಲ್ಲಿ ಒಳಚರಂಡಿ ಚೇಂಬರ್ ಗೆ ಹೊಸದಾಗಿ...,NaN,NaN,Non-corridor,Low,NaN,NaN,NaN,2023-11-11 06:20:00.989398+00,NaN,1,FKUSR00003,FKUSR00001,NaN,NaN,NaN,Wilson Garden,NaN,FKKG000002,NaN,NaN,NaN,FKUSR00003,2024-01-30 04:56:03.281509+00,NaN,NaN,Bengaluru Central Corporation,Central Zone 2,UrvashiJunction
3,FKID000003,unplanned,13.006147,77.579435,13.006239,77.579516,"Sankey Road, Bashyam Circle, Sadashiva Nagar, ...","Sankey Road, Palace Orchard Upper, Sadashiva N...",tree_fall,True,2024-03-07 17:56:55.061+00,NaN,closed,yes,2024-03-14 07:42:05.55005+00,NaN,NaN,tree fall,NaN,NaN,Non-corridor,Low,NaN,NaN,NaN,2024-03-07 17:58:56.696892+00,NaN,1,FKUSR00004,FKUSR00001,NaN,NaN,NaN,Sadashivanagar,NaN,FKKG000003,NaN,NaN,NaN,FKUSR00004,2024-03-14 07:42:05.54944+00,NaN,NaN,NaN,NaN,NaN
4,FKID000004,unplanned,12.953980,77.585233,0.000000,0.000000,"Lalbagh Fort Road, Lalbagh Main Gate Junction,...",NaN,vehicle_breakdown,False,2024-01-30 04:56:32.348+00,NaN,closed,yes,2024-01-30 05:35:17.33908+00,NaN,NaN,[LOCATION] ಪೈಪ್ [PERSON] ವಾಹನ ಆಫ್ ಆಗಿರುತ್ತದೆ ಸರ್,private_bus,FKN00GL0002,Non-corridor,Low,NaN,NaN,NaN,2024-01-30 04:58:55.937662+00,NaN,1,FKUSR00003,FKUSR00001,NaN,NaN,NaN,Wilson Garden,NaN,FKKG000002,NaN,NaN,NaN,FKUSR00003,2024-01-30 05:35:17.338283+00,NaN,NaN,NaN,NaN,LalbaghMainGateJunc


## 3. Standardize Column Names and Null Values

We normalize column names and convert common placeholder nulls like `NULL`, empty strings, and `nan` into real missing values.

In [3]:
df = raw_df.copy()

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
)

NULL_LIKE = {'', ' ', 'NULL', 'null', 'None', 'none', 'NaN', 'nan', 'N/A', 'n/a'}
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].replace(list(NULL_LIKE), np.nan)

print('Cleaned column count:', len(df.columns))
print(df.columns.tolist())

Cleaned column count: 46
['id', 'event_type', 'latitude', 'longitude', 'endlatitude', 'endlongitude', 'address', 'end_address', 'event_cause', 'requires_road_closure', 'start_datetime', 'end_datetime', 'status', 'authenticated', 'modified_datetime', 'map_file', 'direction', 'description', 'veh_type', 'veh_no', 'corridor', 'priority', 'cargo_material', 'reason_breakdown', 'age_of_truck', 'created_date', 'route_path', 'client_id', 'created_by_id', 'last_modified_by_id', 'assigned_to_police_id', 'citizen_accident_id', 'comment', 'police_station', 'meta_data', 'kgid', 'resolved_at_address', 'resolved_at_latitude', 'resolved_at_longitude', 'closed_by_id', 'closed_datetime', 'resolved_by_id', 'resolved_datetime', 'gba_identifier', 'zone', 'junction']


## 4. Basic Schema and Missingness Audit

This gives a quick view of column completeness before deriving labels.

In [4]:
schema_audit = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(df[c].dtype) for c in df.columns],
    'missing_count': [int(df[c].isna().sum()) for c in df.columns],
    'missing_pct': [round(float(df[c].isna().mean() * 100), 2) for c in df.columns],
    'unique_count': [int(df[c].nunique(dropna=True)) for c in df.columns],
})

display(schema_audit.sort_values('missing_pct', ascending=False))

,column,dtype,missing_count,missing_pct,unique_count
34,meta_data,float64,8173,100.00,0
32,comment,float64,8173,100.00,0
15,map_file,float64,8173,100.00,0
16,direction,str,8130,99.47,8
38,resolved_at_longitude,float64,8099,99.09,74
37,resolved_at_latitude,float64,8099,99.09,74
36,resolved_at_address,str,8099,99.09,58
41,resolved_by_id,str,8099,99.09,42
42,resolved_datetime,str,8099,99.09,74
31,citizen_accident_id,str,8045,98.43,77


## 5. Parse Date-Time Columns

Timestamps are parsed as UTC-aware datetimes. We also keep IST versions for local traffic-time features in later notebooks.

In [5]:
datetime_cols = [
    'start_datetime',
    'end_datetime',
    'created_date',
    'modified_datetime',
    'closed_datetime',
    'resolved_datetime',
]

timezone = 'Asia/Kolkata'
for col in datetime_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
        df[f'{col}_ist'] = df[col].dt.tz_convert(timezone)

parsed_summary = pd.DataFrame({
    'datetime_col': [c for c in datetime_cols if c in df.columns],
    'parsed_non_null': [int(df[c].notna().sum()) for c in datetime_cols if c in df.columns],
    'missing_count': [int(df[c].isna().sum()) for c in datetime_cols if c in df.columns],
})
display(parsed_summary)

,datetime_col,parsed_non_null,missing_count
0,start_datetime,8057,116
1,end_datetime,475,7698
2,created_date,8171,2
3,modified_datetime,8173,0
4,closed_datetime,3141,5032
5,resolved_datetime,74,8099


## 6. Clean Coordinates

Start coordinates can be used as prediction-time features. End and resolved coordinates are future/closure-state fields and are not used as model inputs, but we still clean them for audits.

In [6]:
coordinate_cols = [
    'latitude', 'longitude',
    'endlatitude', 'endlongitude',
    'resolved_at_latitude', 'resolved_at_longitude',
]

for col in coordinate_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df.loc[df[col].eq(0), col] = np.nan

# Bengaluru-ish coordinate sanity box. Keep loose bounds to avoid over-cleaning.
if {'latitude', 'longitude'}.issubset(df.columns):
    df['valid_start_coordinate'] = (
        df['latitude'].between(12.0, 14.0) &
        df['longitude'].between(76.0, 78.5)
    ).astype(int)
    df['has_start_location'] = df[['latitude', 'longitude']].notna().all(axis=1).astype(int)

if {'endlatitude', 'endlongitude'}.issubset(df.columns):
    df['has_raw_end_location'] = df[['endlatitude', 'endlongitude']].notna().all(axis=1).astype(int)

if {'resolved_at_latitude', 'resolved_at_longitude'}.issubset(df.columns):
    df['has_resolved_location'] = df[['resolved_at_latitude', 'resolved_at_longitude']].notna().all(axis=1).astype(int)

coord_summary_cols = [c for c in ['valid_start_coordinate', 'has_start_location', 'has_raw_end_location', 'has_resolved_location'] if c in df.columns]
display(df[coord_summary_cols].mean().to_frame('share').round(3))

,share
valid_start_coordinate,1.000
has_start_location,1.000
has_raw_end_location,0.084
has_resolved_location,0.009


## 7. Create Model 1 Label: Road Closure

`requires_road_closure` is converted into a binary target.

In [7]:
def normalize_bool(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    if text in {'true', 't', '1', 'yes', 'y'}:
        return 1
    if text in {'false', 'f', '0', 'no', 'n'}:
        return 0
    return np.nan

if 'requires_road_closure' not in df.columns:
    raise KeyError('requires_road_closure column is required for Model 1 label creation')

df['target_road_closure'] = df['requires_road_closure'].apply(normalize_bool)

print('Road closure target counts:')
display(df['target_road_closure'].value_counts(dropna=False).to_frame('count'))
print('Road closure target percentages:')
display((df['target_road_closure'].value_counts(normalize=True, dropna=False) * 100).round(2).to_frame('percent'))

Road closure target counts:


,count
target_road_closure,
0,7497
1,676


Road closure target percentages:


,percent
target_road_closure,
0,91.73
1,8.27


## 8. Create Duration Candidates

Duration should come from the earliest valid terminal timestamp among:

- `end_datetime`
- `resolved_datetime`
- `closed_datetime`

Then duration is measured from `start_datetime`.

This target is label-only. Terminal timestamps must not be used as model features.

In [8]:
terminal_cols = [c for c in ['end_datetime', 'resolved_datetime', 'closed_datetime'] if c in df.columns]

if 'start_datetime' not in df.columns:
    raise KeyError('start_datetime column is required for duration label creation')

for col in terminal_cols:
    df[f'duration_minutes_from_{col.replace("_datetime", "")}'] = (
        (df[col] - df['start_datetime']).dt.total_seconds() / 60
    )

# Choose the earliest positive terminal timestamp after start.
duration_candidate_cols = [f'duration_minutes_from_{col.replace("_datetime", "")}' for col in terminal_cols]
duration_candidates = df[duration_candidate_cols].where(df[duration_candidate_cols] > 0)
df['duration_minutes'] = duration_candidates.min(axis=1)

source_array = np.full(len(df), np.nan, dtype=object)
for source_col in duration_candidate_cols:
    source_name = source_col.replace('duration_minutes_from_', '')
    matched = duration_candidates[source_col].eq(df['duration_minutes']) & df['duration_minutes'].notna() & pd.isna(source_array)
    source_array[matched.to_numpy()] = source_name

df['duration_source'] = source_array

print('Duration source counts:')
display(df['duration_source'].value_counts(dropna=False).to_frame('count'))
print('Duration minutes summary:')
display(df['duration_minutes'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).to_frame())

Duration source counts:


,count
duration_source,
NaN,4675
closed,3114
end,313
resolved,71


Duration minutes summary:


,duration_minutes
count,3.498000e+03
mean,6.619233e+03
std,4.051182e+04
min,2.855000e-02
10%,1.194402e+01
25%,2.824442e+01
50%,6.969526e+01
75%,7.224332e+02
90%,1.533998e+04
95%,3.692176e+04


## 9. Create Valid Duration Band Label

We keep duration labels only when:

```text
0 < duration_minutes <= 1440
```

The 24-hour cap removes admin-closure noise where records are closed long after the real event impact ended.

In [9]:
df['valid_duration_label'] = df['duration_minutes'].between(0, 1440, inclusive='neither').astype(int)

def duration_to_band(minutes):
    if pd.isna(minutes) or minutes <= 0 or minutes > 1440:
        return np.nan
    if minutes < 60:
        return 'short'
    if minutes <= 240:
        return 'medium'
    return 'long'

df['duration_band'] = df['duration_minutes'].apply(duration_to_band)

print('Valid duration rows:', int(df['valid_duration_label'].sum()), '/', len(df))
print('\nDuration band counts:')
display(df['duration_band'].value_counts(dropna=False).to_frame('count'))
print('\nDuration band percentages among valid labels:')
display((df.loc[df['valid_duration_label'].eq(1), 'duration_band'].value_counts(normalize=True) * 100).round(2).to_frame('percent'))

Valid duration rows: 2764 / 8173

Duration band counts:


,count
duration_band,
NaN,5409
short,1581
medium,903
long,280



Duration band percentages among valid labels:


,percent
duration_band,
short,57.20
medium,32.67
long,10.13


## 10. Create Event-Creation Time Audit Features

These are safe helper fields created from event creation/start time. Later feature engineering can reuse them.

In [10]:
if {'created_date', 'start_datetime'}.issubset(df.columns):
    df['report_lag_minutes'] = (df['created_date'] - df['start_datetime']).dt.total_seconds() / 60
    df['report_lag_is_negative'] = (df['report_lag_minutes'] < 0).astype(int)

if 'start_datetime_ist' in df.columns:
    df['start_hour'] = df['start_datetime_ist'].dt.hour
    df['start_dayofweek'] = df['start_datetime_ist'].dt.dayofweek
    df['start_month'] = df['start_datetime_ist'].dt.to_period('M').astype(str)
    df['is_weekend'] = df['start_dayofweek'].isin([5, 6]).astype(int)
    df['is_peak_hour'] = df['start_hour'].isin([7, 8, 9, 17, 18, 19, 20]).astype(int)
    df['is_night'] = df['start_hour'].isin([22, 23, 0, 1, 2, 3, 4, 5]).astype(int)

audit_cols = [c for c in ['report_lag_minutes', 'report_lag_is_negative', 'start_hour', 'start_dayofweek', 'start_month', 'is_weekend', 'is_peak_hour', 'is_night'] if c in df.columns]
display(df[audit_cols].head())

,report_lag_minutes,report_lag_is_negative,start_hour,start_dayofweek,start_month,is_weekend,is_peak_hour,is_night
0,2.050884,0,22.0,3.0,2024-03,0,0,1
1,0.979700,0,9.0,1.0,2024-01,0,1,0
2,1.960773,0,11.0,5.0,2023-11,1,0,0
3,2.027265,0,23.0,3.0,2024-03,0,0,1
4,2.393161,0,10.0,1.0,2024-01,0,0,0


## 11. Leakage Policy

This table documents which columns are safe as prediction-time model inputs and which should only be used for label creation/auditing.

In [11]:
future_or_leakage_cols = {
    'status',
    'modified_datetime', 'modified_datetime_ist',
    'end_datetime', 'end_datetime_ist',
    'closed_datetime', 'closed_datetime_ist',
    'resolved_datetime', 'resolved_datetime_ist',
    'closed_by_id', 'resolved_by_id', 'last_modified_by_id',
    'resolved_at_address', 'resolved_at_latitude', 'resolved_at_longitude',
    'endlatitude', 'endlongitude', 'end_address', 'route_path',
    'duration_minutes', 'duration_band', 'duration_source', 'valid_duration_label',
}

id_or_admin_cols = {
    'id', 'veh_no', 'map_file', 'client_id', 'created_by_id', 'assigned_to_police_id',
    'citizen_accident_id', 'meta_data', 'kgid', 'gba_identifier', 'comment',
}

target_cols = {'target_road_closure', 'requires_road_closure'}

policy_rows = []
for col in df.columns:
    if col in target_cols:
        role = 'target_or_raw_target'
        use_as_feature = False
    elif col in future_or_leakage_cols or col.startswith('duration_minutes_from_'):
        role = 'future_or_label_only'
        use_as_feature = False
    elif col in id_or_admin_cols:
        role = 'id_or_admin'
        use_as_feature = False
    else:
        role = 'prediction_time_candidate'
        use_as_feature = True
    policy_rows.append({'column': col, 'role': role, 'use_as_model_feature': use_as_feature})

leakage_policy = pd.DataFrame(policy_rows)
display(leakage_policy.groupby(['role', 'use_as_model_feature']).size().reset_index(name='columns'))
display(leakage_policy)

,role,use_as_model_feature,columns
0,future_or_label_only,False,26
1,id_or_admin,False,11
2,prediction_time_candidate,True,33
3,target_or_raw_target,False,2


,column,role,use_as_model_feature
0,id,id_or_admin,False
1,event_type,prediction_time_candidate,True
2,latitude,prediction_time_candidate,True
3,longitude,prediction_time_candidate,True
4,endlatitude,future_or_label_only,False
5,endlongitude,future_or_label_only,False
6,address,prediction_time_candidate,True
7,end_address,future_or_label_only,False
8,event_cause,prediction_time_candidate,True
9,requires_road_closure,target_or_raw_target,False


## 12. Save Cleaned Label Dataset

This output is the contract for the next block: feature engineering.

In [12]:
cleaned_path = OUTPUT_DIR / 'cleaned_events_with_labels.csv'
label_summary_path = OUTPUT_DIR / 'label_summary.csv'
leakage_policy_path = OUTPUT_DIR / 'leakage_policy.csv'

label_summary = pd.DataFrame({
    'metric': [
        'raw_rows',
        'raw_columns',
        'cleaned_columns',
        'road_closure_labeled_rows',
        'road_closure_positive_rows',
        'valid_duration_rows',
        'duration_short_rows',
        'duration_medium_rows',
        'duration_long_rows',
    ],
    'value': [
        len(df),
        len(raw_df.columns),
        len(df.columns),
        int(df['target_road_closure'].notna().sum()),
        int(df['target_road_closure'].eq(1).sum()),
        int(df['valid_duration_label'].sum()),
        int(df['duration_band'].eq('short').sum()),
        int(df['duration_band'].eq('medium').sum()),
        int(df['duration_band'].eq('long').sum()),
    ]
})

df.to_csv(cleaned_path, index=False)
label_summary.to_csv(label_summary_path, index=False)
leakage_policy.to_csv(leakage_policy_path, index=False)

print('Saved cleaned dataset:', cleaned_path)
print('Saved label summary:', label_summary_path)
print('Saved leakage policy:', leakage_policy_path)
display(label_summary)

Saved cleaned dataset: /Users/astron_designer/GridLock_Phase2/outputs/cleaned/cleaned_events_with_labels.csv
Saved label summary: /Users/astron_designer/GridLock_Phase2/outputs/cleaned/label_summary.csv
Saved leakage policy: /Users/astron_designer/GridLock_Phase2/outputs/cleaned/leakage_policy.csv


,metric,value
0,raw_rows,8173
1,raw_columns,46
2,cleaned_columns,72
3,road_closure_labeled_rows,8173
4,road_closure_positive_rows,676
5,valid_duration_rows,2764
6,duration_short_rows,1581
7,duration_medium_rows,903
8,duration_long_rows,280


## 13. Block 1 Completion Checklist

Before moving to Block 2, verify:

- `target_road_closure` exists and has valid binary labels.
- `duration_band` exists for valid duration rows only.
- Future columns are documented in `leakage_policy.csv`.
- The cleaned CSV is saved under `outputs/cleaned/`.

In [13]:
required_cols = ['target_road_closure', 'duration_minutes', 'valid_duration_label', 'duration_band']
missing_required = [c for c in required_cols if c not in df.columns]
assert not missing_required, f'Missing required columns: {missing_required}'
assert cleaned_path.exists(), cleaned_path
assert label_summary_path.exists(), label_summary_path
assert leakage_policy_path.exists(), leakage_policy_path

print('Block 1 complete.')
print('Next block: Feature engineering from outputs/cleaned/cleaned_events_with_labels.csv')

Block 1 complete.
Next block: Feature engineering from outputs/cleaned/cleaned_events_with_labels.csv
